# Build the RAG index on Kaggle

Builds `artifacts/chroma` + `artifacts/chunks.jsonl` from the **rag-data** input, then gives you `artifacts.zip` to copy into the `Code` folder on the Orange Pi.

**Before running:** right panel → Settings → **Accelerator: GPU** and **Internet: On**. Then *Run All*.

In [ ]:
!pip install -q pypdf cryptography rank_bm25 langchain-chroma langchain-huggingface langchain-text-splitters sentence-transformers

In [ ]:
# Find the rag-data input and link it as "RAG Data" (the folder name rag.py expects)
import glob, os
%cd /kaggle/working
src = next(p for p in glob.glob("/kaggle/input/**/rag-data", recursive=True) if os.path.isdir(p))
if os.path.islink("RAG Data"):
    os.remove("RAG Data")
os.symlink(src, "RAG Data")
pdfs = glob.glob("RAG Data/**/*.pdf", recursive=True)
print(src, "->", len(pdfs), "PDFs")

In [ ]:
%%writefile rag.py
"""
RAG: PDFs in "RAG Data" -> chunks -> Chroma (bge-small) + BM25 -> reciprocal rank fusion -> bge reranker.

    python rag.py      # build the index (once, or when PDFs change)
"""
import json
import logging
import re
from functools import lru_cache
from pathlib import Path

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from rank_bm25 import BM25Okapi

DOCS = Path("RAG Data")
CHROMA = "artifacts/chroma"
CHUNKS = Path("artifacts/chunks.jsonl")
logging.getLogger("pypdf").setLevel(logging.ERROR)


def embedder(progress=False):
    return HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5", show_progress=progress,
                                 encode_kwargs={"normalize_embeddings": True})


def tokens(text):
    return re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", text.lower())   # keeps pm2.5, fev1, no2 whole


def page_text(page):
    try:
        return re.sub(r"\s+", " ", page.extract_text() or "").strip()
    except Exception:                                            # broken pages in some govt PDFs
        return ""


def read_pdf(f):
    try:
        return [Document(page_content=t, metadata={"source": f.name, "page": i + 1})
                for i, p in enumerate(PdfReader(f).pages) if len(t := page_text(p)) >= 50]
    except Exception as e:                                       # e.g. encrypted / corrupt file
        print(f"skipped {f.name}: {e}")
        return []


def build():
    pages = [d for f in sorted(DOCS.rglob("*.pdf")) for d in read_pdf(f)]
    chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120).split_documents(pages)
    ids = [f"c{i}" for i in range(len(chunks))]
    for i, c in zip(ids, chunks):
        c.metadata["id"] = i
    CHUNKS.parent.mkdir(exist_ok=True)
    CHUNKS.write_text("\n".join(json.dumps({"text": c.page_content, **c.metadata}) for c in chunks), encoding="utf-8")
    Chroma.from_documents(chunks, embedder(progress=True), ids=ids, collection_name="copd", persist_directory=CHROMA,
                          collection_metadata={"hnsw:space": "cosine"})
    print(f"Indexed {len(chunks)} chunks")


class Retriever:
    def __init__(self):
        from sentence_transformers import CrossEncoder
        self.chunks = {d["id"]: d for d in map(json.loads, CHUNKS.read_text(encoding="utf-8").splitlines())}
        self.ids = list(self.chunks)
        self.bm25 = BM25Okapi([tokens(self.chunks[i]["text"]) for i in self.ids])
        self.db = Chroma(collection_name="copd", embedding_function=embedder(), persist_directory=CHROMA)
        self.reranker = CrossEncoder("BAAI/bge-reranker-base")

    @lru_cache(maxsize=None)              # same query -> same passages; saves a lot of CPU in validation
    def retrieve(self, query, k=20, top=4):
        dense = [d.metadata["id"] for d in self.db.similarity_search(query, k=k)]
        scores = self.bm25.get_scores(tokens(query))
        sparse = [self.ids[i] for i in scores.argsort()[::-1][:k] if scores[i] > 0]
        fused = {}
        for ranking in (dense, sparse):                          # reciprocal rank fusion
            for r, cid in enumerate(ranking):
                fused[cid] = fused.get(cid, 0) + 1 / (61 + r)
        cands = list(fused)
        ce = self.reranker.predict([(query, self.chunks[c]["text"]) for c in cands])
        best = sorted(zip(ce, cands), reverse=True)[:top]
        return tuple(self.chunks[c] for _, c in best)


if __name__ == "__main__":
    build()


In [ ]:
!rm -rf artifacts
!python rag.py

In [ ]:
# Zip the index and show the versions the Orange Pi must match
!zip -qr artifacts.zip artifacts
!pip show chromadb langchain-chroma | grep -E "^(Name|Version)"
from IPython.display import FileLink
FileLink("artifacts.zip")

## On the Orange Pi (inside the `Code` folder)
```fish
pip install --no-cache-dir chromadb==<Version printed above>
rm -rf artifacts/chroma artifacts/chunks.jsonl
unzip -o artifacts.zip
python experiments.py
```